In [2]:
import os
import base64
import re
import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


username = "aacuser"
password = "Builderpro123"

# Connect to the database through the CRUD module.
db = AnimalShelter(username, password)

# Force a real connection check so authentication errors are visible.
db.client.admin.command("ping")

# The preferred breeds, sexes and age ranges come from the project specifications.
# A suffix such as 'Mix' is allowed, since many AAC breeds include that suffix.
RESCUE_FILTERS = {
    "water": {
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female", "min_age": 26, "max_age": 156,
    },
    "mountain": {
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                   "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male", "min_age": 26, "max_age": 156,
    },
    "disaster": {
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                   "Bloodhound", "Rottweiler"],
        "sex": "Intact Male", "min_age": 20, "max_age": 300,
    },
}


def mongo_query(filter_type):
    """Build the MongoDB query for the rescue type selected by the user."""
    if filter_type not in RESCUE_FILTERS:
        return {}  # Reset: show the entire animal collection.
    settings = RESCUE_FILTERS[filter_type]
    # Escape breed names before using them in a case-insensitive regex.
    # Prefix matching accommodates entries such as 'German Shepherd Mix'.
    patterns = ["^" + re.escape(breed) + r"(?: Mix|/.*)?$" for breed in settings["breeds"]]
    return {
        "animal_type": "Dog",
        "breed": {"$regex": "(?:" + "|".join(patterns) + ")", "$options": "i"},
        "sex_upon_outcome": settings["sex"],
        "age_upon_outcome_in_weeks": {
            "$gte": settings["min_age"], "$lte": settings["max_age"]
        },
    }


def fetch_table(filter_type):
    """Read matching animals from MongoDB and prepare rows for the Dash table."""
    records = db.read(mongo_query(filter_type))
    table = pd.DataFrame.from_records(records)
    table.drop(columns=["_id"], inplace=True, errors="ignore")
    # Keep dates and other values serializable for Dash.
    for col in table.columns:
        table[col] = table[col].where(table[col].notna(), None)
    return table.to_dict("records")


# Read all records to establish the columns and initial, unfiltered view.
initial_data = fetch_table("reset")
if not initial_data:
    raise RuntimeError("No AAC records were returned. Check the database and authentication.")
columns = [{"name": key, "id": key, "selectable": True} for key in initial_data[0].keys()]

# Display the provided Grazioso Salvare logo and link it to the requested website.
logo_filename = "Grazioso_Salvare_Logo.png"
if not os.path.exists(logo_filename):
    raise FileNotFoundError(
        "Place 'Grazioso Salvare Logo.png' in the same code_files folder as this notebook."
    )
with open(logo_filename, "rb") as logo_file:
    encoded_logo = base64.b64encode(logo_file.read()).decode("utf-8")

app = JupyterDash(__name__)
app.layout = html.Div([
    html.Div([
        html.A(html.Img(src="data:image/png;base64," + encoded_logo,
                            style={"height": "110px", "maxWidth": "100%", "objectFit": "contain"}),
               href="https://www.snhu.edu", target="_blank"),
        html.Div([
            html.H1("Grazioso Salvare Rescue Dashboard", style={"marginBottom": "6px"}),
            html.P("CS 340 Project Two | Created by Jonathan Streetman")
        ])
    ], style={"display": "flex", "alignItems": "center", "gap": "20px"}),
    html.Hr(),
    html.H3("Rescue training filters"),
    dcc.RadioItems(
        id="filter-type", value="reset",
        options=[
            {"label": "Reset / All Animals", "value": "reset"},
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster or Individual Tracking", "value": "disaster"},
        ],
        labelStyle={"display": "inline-block", "marginRight": "18px", "marginBottom": "12px"}
    ),
    html.Div(id="record-count", style={"marginBottom": "10px"}),
    dash_table.DataTable(
        id="datatable-id", columns=columns, data=initial_data,
        page_size=10, page_action="native", sort_action="native",
        filter_action="native", row_selectable="single",
        selected_rows=[0], selected_columns=[],
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "115px", "maxWidth": "240px",
                    "overflow": "hidden", "textOverflow": "ellipsis", "padding": "6px"},
        style_header={"fontWeight": "bold", "backgroundColor": "#eeeeee"},
    ),
    html.Br(),
    html.Div([
        html.Div(id="graph-id", style={"flex": "1", "minWidth": "320px"}),
        html.Div(id="map-id", style={"flex": "1", "minWidth": "320px"})
    ], style={"display": "flex", "flexWrap": "wrap", "gap": "18px"})
], style={"padding": "18px", "fontFamily": "Arial, sans-serif"})


# Controller: a filter change retrieves fresh matching records from MongoDB.
@app.callback(
    [Output("datatable-id", "data"), Output("datatable-id", "selected_rows"),
     Output("record-count", "children")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    rows = fetch_table(filter_type)
    return rows, ([0] if rows else []), f"Matching animal records: {len(rows)}"


# Both charts use the currently visible table data, so filtering and sorting
# in the DataTable also affect what is displayed.
@app.callback(Output("graph-id", "children"),
              [Input("datatable-id", "derived_virtual_data"),
               Input("datatable-id", "data")])
def update_graphs(visible_rows, table_rows):
    rows = visible_rows if visible_rows is not None else table_rows
    if not rows:
        return html.P("No animal records match this selection.")
    chart_df = pd.DataFrame(rows)
    counts = chart_df["breed"].fillna("Unknown").value_counts().head(10).reset_index()
    counts.columns = ["breed", "count"]
    fig = px.pie(counts, names="breed", values="count", title="Top 10 Breeds in Current Results")
    fig.update_layout(margin=dict(l=10, r=10, t=55, b=10), legend=dict(font=dict(size=10)))
    return dcc.Graph(figure=fig, config={"displayModeBar": False})


@app.callback(Output("datatable-id", "style_data_conditional"),
              [Input("datatable-id", "selected_columns")])
def update_styles(selected_columns):
    return [{"if": {"column_id": column}, "backgroundColor": "#D2F3FF"}
            for column in (selected_columns or [])]


@app.callback(Output("map-id", "children"),
              [Input("datatable-id", "derived_virtual_data"),
               Input("datatable-id", "derived_virtual_selected_rows"),
               Input("datatable-id", "data")])
def update_map(visible_rows, selected_rows, table_rows):
    rows = visible_rows if visible_rows is not None else table_rows
    if not rows:
        return html.P("No location is available for this selection.")
    # Use the selected row if available; otherwise default to the first row.
    index = selected_rows[0] if selected_rows and selected_rows[0] < len(rows) else 0
    record = rows[index]
    try:
        latitude = float(record.get("location_lat"))
        longitude = float(record.get("location_long"))
        if not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
            raise ValueError("Invalid coordinates")
    except (TypeError, ValueError):
        return html.P("The selected animal does not have valid map coordinates.")
    name = str(record.get("name") or "Unnamed animal")
    breed = str(record.get("breed") or "Unknown breed")
    return dl.Map(
        style={"width": "100%", "height": "420px"},
        center=[latitude, longitude], zoom=11,
        children=[
            dl.TileLayer(),
            dl.Marker(position=[latitude, longitude], children=[
                dl.Tooltip(breed),
                dl.Popup([html.H4(name), html.P(breed)])
            ])
        ]
    )


# Start the dashboard inside Jupyter Lab.
app.run_server(mode="inline")
